Reiniciar python


In [0]:
%pip install apify-client
dbutils.library.restartPython()


Setup variables


In [0]:
import os
os.environ["GAIN_CATALOG"] = "genesis_devp"
os.environ["GAIN_SCHEMA"]  = "dev_ricardo"

##
os.environ["GAIN_FETCH_BATCH_SIZE"]   = "3"    # menos terminos por llamada, no se agota el tiempo
os.environ["GAIN_MAX_ARTICLES"]       = "5"    # menos articulos por termino
os.environ["GAIN_FETCH_WINDOW_DAYS"]  = "3"    # ventana mas corta (default 10)
##

import sys, importlib
sys.path.insert(0, "/Workspace/Users/rhernandez@genesisig.com")

config = importlib.import_module("genesis-gain.config")
assert config.CATALOG == "genesis_devp" and config.SCHEMA == "dev_ricardo", \
    f"APUNTANDO MAL a {config.CATALOG}.{config.SCHEMA}"
print("apuntando a", config.CATALOG + "." + config.SCHEMA)


Diag


In [0]:
context = importlib.import_module("genesis-gain.context")
fetch   = importlib.import_module("genesis-gain.fetch")

print("PASO 1  leer el secret")
token = context.secret(config.APIFY_TOKEN_KEY)
print(f"        OK — {len(token)} caracteres, termina en ...{token[-4:]}")

print("\nPASO 2  crear el cliente de Apify")
client = context.apify()
print("        OK —", client)

print("\nPASO 3  una llamada real al actor")
queries = fetch.registry_queries("policy")
print(f"        {len(queries)} terminos; ejemplo: {queries[0]}")
out = fetch.fetch_batch(queries[:2], 3, "policy")
print(f"        articulos devueltos: {len(out)}")
for a in out[:3]:
    print("        -", str(a.get("title"))[:70])


PipeLine


In [0]:
orchestrator = importlib.import_module("genesis-gain.orchestrator")
report = orchestrator.run_pipeline(use_cache=True, limit_articles=5)

import json
print(json.dumps(report, indent=2, default=str))


Vlidations

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM genesis_devp.dev_ricardo.dra_findings)       AS findings,
  (SELECT COUNT(*) FROM genesis_devp.dev_ricardo.dra_feature_vector) AS feature_vector,
  (SELECT COUNT(*) FROM genesis_devp.dev_ricardo.dra_feature_index)  AS feature_index
